# SI Demo — Music Knowledge Graph

**Domain:** 15 albums · artist · genre · release year · record label

**Skills demonstrated in this notebook:**

| # | Skill | Shown in |
|---|-------|---------|
| 1 | Turtle / RDF syntax | `MUSIC_TTL` block |
| 2 | SKOS `prefLabel` / `altLabel` | Genre concept definitions |
| 3 | `SELECT`, triple patterns, `ORDER BY` | All three queries |
| 4 | Property-path `(a\|b)` disambiguation | Q2 — rock vs "rock and roll" |
| 5 | `FILTER` with year comparison + `\|\|` | Q3 — post-2000 hip-hop |
| 6 | `rdflib` in-memory SPARQL engine | Local cells (no server needed) |
| 7 | Apache Fuseki 5 via Docker | Fuseki cells |
| 8 | rdflib-vs-Fuseki cross-check | Cross-check cell |

> **Instructor note:** This demo uses the *music* domain only.
> The student lab uses a separate *recipes* domain — no overlap, no spoilers.

---

In [1]:
import subprocess, sys                                      # stdlib: run external commands; access current interpreter info

subprocess.run(                                             # launch a subprocess and wait for completion
    [sys.executable, "-m", "pip", "install",               # invoke the active Python's pip as a module
     "rdflib", "SPARQLWrapper", "requests", "pandas", "-q"],  # packages to install; -q suppresses verbose progress
    check=True,                                             # raise CalledProcessError if pip exits non-zero
)
print("Dependencies ready.")                                # confirm to the user that all imports will succeed

Dependencies ready.


In [2]:
import subprocess                                           # spawn external shell processes (Docker, pip)
import time                                                 # sleep between Fuseki health-poll attempts
from pathlib import Path                                    # cross-platform, object-oriented filesystem paths

import pandas as pd                                         # tabular data display for SPARQL result sets
import requests                                             # HTTP client for Fuseki admin API and Graph Store Protocol
import rdflib                                               # in-memory RDF graph with a SPARQL 1.1 engine
from SPARQLWrapper import JSON, SPARQLWrapper               # JSON result-format flag; wrapper for remote SPARQL endpoints

NOTEBOOK_DIR = Path.cwd()                                   # path of the directory where this notebook was launched
FUSEKI_BASE  = "http://localhost:3030"                      # base URL of the Fuseki server exposed by Docker
DATASET      = "music"                                      # name of the Fuseki dataset to create and query
ENDPOINT     = f"{FUSEKI_BASE}/{DATASET}/sparql"            # full SPARQL query URL used by SPARQLWrapper
ADMIN        = ("admin", "admin")                           # (username, password) for Fuseki admin HTTP auth

# Shared PREFIX declarations prepended to every SPARQL query
PREFIXES = """
PREFIX mu:   <http://example.org/music/>                    # shorthand alias for the custom music namespace IRI
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>         # SKOS vocabulary: prefLabel, altLabel, Concept
PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>            # XML Schema types: integer, string, date
"""


def local_query(query_body: str) -> pd.DataFrame:
    """Run SPARQL against the in-memory rdflib graph; return a DataFrame."""
    result = G.query(PREFIXES + query_body)                 # prepend shared prefixes, then execute on the global graph G
    cols = [str(v) for v in result.vars]                    # pull variable names to use as column headers
    rows = [[str(v) if v is not None else "" for v in row] for row in result]  # stringify each cell; unbound vars → ""
    return pd.DataFrame(rows, columns=cols)                 # wrap rows in a labeled pandas DataFrame


def fuseki_query(query_body: str) -> pd.DataFrame:
    """Run SPARQL against the Fuseki SPARQL endpoint; return a DataFrame."""
    sp = SPARQLWrapper(ENDPOINT)                            # create a wrapper pointed at the running Fuseki URL
    sp.setQuery(PREFIXES + query_body)                      # attach shared prefixes before sending the query
    sp.setReturnFormat(JSON)                                # request application/sparql-results+json from the server
    bindings = sp.query().convert()["results"]["bindings"]  # fire the HTTP request; drill into the result rows list
    if not bindings:                                        # empty result — guard against KeyError on bindings[0]
        return pd.DataFrame()                               # return empty DataFrame so callers need no special handling
    cols = list(bindings[0].keys())                         # extract column names from the first row's variable keys
    rows = [[b.get(c, {}).get("value", "") for c in cols] for b in bindings]  # flatten each binding dict to plain strings
    return pd.DataFrame(rows, columns=cols)                 # wrap results in a labeled pandas DataFrame

/usr/lib/python3/dist-packages/pytz/__init__.py:31: SyntaxWarning: invalid escape sequence '\s'
  match = re.match("^#\s*version\s*([0-9a-z]*)\s*$", line)


## 1 · The Turtle Knowledge Graph

[Turtle](https://www.w3.org/TR/turtle/) (`.ttl`) is the most readable RDF serialization.
Every statement is a **subject → predicate → object** triple terminated by `.`

```turtle
@prefix mu: <http://example.org/music/> .   ← bind a short alias to a base IRI
mu:Rock  a  skos:Concept ;                  ← type declaration  (a = rdf:type)
         skos:prefLabel "rock" ;             ← preferred label (one per language)
         skos:altLabel  "rock and roll" .    ← synonym/variant label
```

**SKOS** (Simple Knowledge Organization System) is the W3C standard for controlled
vocabularies. Storing `"rock and roll"` as an `altLabel` of the `mu:Rock` concept lets
a single SPARQL query find albums regardless of which label term was used — demonstrated in Q2.

Run the cell below to parse the TTL string into an rdflib in-memory graph (`G`).

In [3]:
MUSIC_TTL = """
@prefix mu:   <http://example.org/music/> .
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .
@prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .

# ── Genre concepts (SKOS) ────────────────────────────────────────────────────

mu:Rock a skos:Concept ;
    skos:prefLabel "rock" ;
    skos:altLabel  "rock and roll" ,
                   "rock music" .

mu:HipHop a skos:Concept ;
    skos:prefLabel "hip-hop" ;
    skos:altLabel  "rap" ,
                   "hip hop" .

mu:Jazz a skos:Concept ;
    skos:prefLabel "jazz" ;
    skos:altLabel  "jazz music" .

mu:Electronic a skos:Concept ;
    skos:prefLabel "electronic" ;
    skos:altLabel  "EDM" ,
                   "electronica" .

mu:RnB a skos:Concept ;
    skos:prefLabel "R&B" ;
    skos:altLabel  "rhythm and blues" ,
                   "soul" .

# ── Albums ───────────────────────────────────────────────────────────────────

mu:nevermind a mu:Album ;
    mu:title       "Nevermind" ;
    mu:artist      "Nirvana" ;
    mu:genre       mu:Rock ;
    mu:releaseYear 1991 ;
    mu:label       "DGC Records" .

mu:darkside a mu:Album ;
    mu:title       "The Dark Side of the Moon" ;
    mu:artist      "Pink Floyd" ;
    mu:genre       mu:Rock ;
    mu:releaseYear 1973 ;
    mu:label       "Harvest Records" .

mu:okcomputer a mu:Album ;
    mu:title       "OK Computer" ;
    mu:artist      "Radiohead" ;
    mu:genre       mu:Rock ;
    mu:releaseYear 1997 ;
    mu:label       "Parlophone" .

mu:abbeyroad a mu:Album ;
    mu:title       "Abbey Road" ;
    mu:artist      "The Beatles" ;
    mu:genre       mu:Rock ;
    mu:releaseYear 1969 ;
    mu:label       "Apple Records" .

mu:ledzeppeliniv a mu:Album ;
    mu:title       "Led Zeppelin IV" ;
    mu:artist      "Led Zeppelin" ;
    mu:genre       mu:Rock ;
    mu:releaseYear 1971 ;
    mu:label       "Atlantic Records" .

mu:thechronic a mu:Album ;
    mu:title       "The Chronic" ;
    mu:artist      "Dr. Dre" ;
    mu:genre       mu:HipHop ;
    mu:releaseYear 1992 ;
    mu:label       "Death Row Records" .

mu:topimpabutterfly a mu:Album ;
    mu:title       "To Pimp a Butterfly" ;
    mu:artist      "Kendrick Lamar" ;
    mu:genre       mu:HipHop ;
    mu:releaseYear 2015 ;
    mu:label       "Top Dawg Entertainment" .

mu:takecare a mu:Album ;
    mu:title       "Take Care" ;
    mu:artist      "Drake" ;
    mu:genre       mu:HipHop ;
    mu:releaseYear 2011 ;
    mu:label       "Young Money / Cash Money" .

mu:theblueprint a mu:Album ;
    mu:title       "The Blueprint" ;
    mu:artist      "Jay-Z" ;
    mu:genre       mu:HipHop ;
    mu:releaseYear 2001 ;
    mu:label       "Roc-A-Fella Records" .

mu:kindofblue a mu:Album ;
    mu:title       "Kind of Blue" ;
    mu:artist      "Miles Davis" ;
    mu:genre       mu:Jazz ;
    mu:releaseYear 1959 ;
    mu:label       "Columbia Records" .

mu:alovesupreme a mu:Album ;
    mu:title       "A Love Supreme" ;
    mu:artist      "John Coltrane" ;
    mu:genre       mu:Jazz ;
    mu:releaseYear 1965 ;
    mu:label       "Impulse! Records" .

mu:randomaccess a mu:Album ;
    mu:title       "Random Access Memories" ;
    mu:artist      "Daft Punk" ;
    mu:genre       mu:Electronic ;
    mu:releaseYear 2013 ;
    mu:label       "Columbia Records" .

mu:discovery a mu:Album ;
    mu:title       "Discovery" ;
    mu:artist      "Daft Punk" ;
    mu:genre       mu:Electronic ;
    mu:releaseYear 2001 ;
    mu:label       "Virgin Records" .

mu:thriller a mu:Album ;
    mu:title       "Thriller" ;
    mu:artist      "Michael Jackson" ;
    mu:genre       mu:RnB ;
    mu:releaseYear 1982 ;
    mu:label       "Epic Records" .

mu:purplerain a mu:Album ;
    mu:title       "Purple Rain" ;
    mu:artist      "Prince" ;
    mu:genre       mu:RnB ;
    mu:releaseYear 1984 ;
    mu:label       "Warner Bros. Records" .
"""

G = rdflib.Graph()
G.parse(data=MUSIC_TTL, format="turtle")
print(f"Graph loaded — {len(G)} triples")

Graph loaded — 109 triples


## 2 · Query 1 — All Albums

The simplest query: bind every `mu:Album` node and pull three properties off it.

```sparql
SELECT ?title ?artist ?year WHERE {
    ?album a mu:Album ;           -- ?album must have type mu:Album
           mu:title  ?title ;     -- ; continues predicates for the same subject
           mu:artist ?artist ;
           mu:releaseYear ?year .
}
ORDER BY ?year                    -- sort oldest → newest
```

`a` is RDF shorthand for `rdf:type`.
Every `?variable` becomes a column in the result table.
`ORDER BY` sorts rows by the bound value of `?year` using numeric ordering.

In [4]:
Q1 = """
SELECT ?title ?artist ?year WHERE {
    ?album a mu:Album ;
           mu:title       ?title ;
           mu:artist      ?artist ;
           mu:releaseYear ?year .
}
ORDER BY ?year
"""

local_query(Q1)

,title,artist,year
0,Kind of Blue,Miles Davis,1959
1,A Love Supreme,John Coltrane,1965
2,Abbey Road,The Beatles,1969
3,Led Zeppelin IV,Led Zeppelin,1971
4,The Dark Side of the Moon,Pink Floyd,1973
5,Thriller,Michael Jackson,1982
6,Purple Rain,Prince,1984
7,Nevermind,Nirvana,1991
8,The Chronic,Dr. Dre,1992
9,OK Computer,Radiohead,1997


## 3 · Query 2 — Rock Albums via SKOS Disambiguation

**The key technique:** a SPARQL 1.1 **property path** traverses both `skos:prefLabel`
and `skos:altLabel` in a single triple pattern.

```sparql
?genre (skos:prefLabel|skos:altLabel) ?label .
        ───────────────┬──────────────
                       └─ alternation path — "either predicate"
```

`FILTER(CONTAINS(LCASE(STR(?label)), "rock"))` then checks every matched label
for the substring `"rock"` (case-insensitive via `LCASE`).

This matches `"rock"`, `"rock and roll"`, and `"rock music"` from the same concept node,
so the query finds albums whether the genre was labelled with the preferred term or a synonym.

`SELECT DISTINCT` collapses duplicates that arise because each of the 5 rock albums
matches all 3 rock-related labels — producing (title, matched-label) pairs.

In [5]:
Q2 = """
SELECT DISTINCT ?title ?label WHERE {
    ?album a mu:Album ;
           mu:title ?title ;
           mu:genre ?genre .
    ?genre (skos:prefLabel|skos:altLabel) ?label .
    FILTER(CONTAINS(LCASE(STR(?label)), "rock"))
}
ORDER BY ?title
"""

local_query(Q2)

,title,label
0,Abbey Road,rock
1,Abbey Road,rock and roll
2,Abbey Road,rock music
3,Led Zeppelin IV,rock
4,Led Zeppelin IV,rock and roll
5,Led Zeppelin IV,rock music
6,Nevermind,rock
7,Nevermind,rock and roll
8,Nevermind,rock music
9,OK Computer,rock


## 4 · Query 3 — Hip-Hop / Rap Albums Released ≥ 2000

Two `FILTER`s narrow the results:

**Year filter**
```sparql
FILTER(?year >= 2000)
```
`?year` is an `xsd:integer` literal — SPARQL numeric ordering applies directly.
*The Chronic* (1992) is correctly excluded without any string parsing.

**Genre filter with `||`**
```sparql
FILTER(
    CONTAINS(LCASE(STR(?label)), "hip") ||
    CONTAINS(LCASE(STR(?label)), "rap")
)
```
`||` is logical OR — either condition alone is sufficient.
`"hip"` catches both `"hip-hop"` and `"hip hop"`; `"rap"` catches the `altLabel "rap"`.

`SELECT DISTINCT` prevents duplicate rows from multiple matching labels.

In [6]:
Q3 = """
SELECT DISTINCT ?title ?year WHERE {
    ?album a mu:Album ;
           mu:title       ?title ;
           mu:genre       ?genre ;
           mu:releaseYear ?year .
    ?genre (skos:prefLabel|skos:altLabel) ?label .
    FILTER(?year >= 2000)
    FILTER(
        CONTAINS(LCASE(STR(?label)), "hip") ||
        CONTAINS(LCASE(STR(?label)), "rap")
    )
}
ORDER BY ?year
"""

local_query(Q3)

,title,year
0,The Blueprint,2001
1,Take Care,2011
2,To Pimp a Butterfly,2015


## 5 · Launch Apache Fuseki

The cell below does three things automatically:

1. **Writes** a minimal `docker-compose.yml` to the notebook directory
2. **Starts** Fuseki in the background with `docker compose up -d`
3. **Polls** `/$/ping` every 2 s until the server is healthy (up to 60 s)

**Prerequisites:**
- Docker Desktop (or Docker Engine) is running
- `docker` is on your `PATH`
- First run pulls `stain/jena-fuseki:5.0.0` (~3 min); subsequent runs start in seconds

In [7]:
COMPOSE_YML = """
version: '3.8'
services:
  fuseki:
    image: stain/jena-fuseki:5.0.0
    ports:
      - "3030:3030"
    environment:
      - ADMIN_PASSWORD=admin
    volumes:
      - fuseki_data:/fuseki
volumes:
  fuseki_data:
"""

compose_path = NOTEBOOK_DIR / "docker-compose.yml"
compose_path.write_text(COMPOSE_YML, encoding="utf-8")
print(f"Wrote {compose_path}")

up = subprocess.run(
    ["docker", "compose", "up", "-d"],
    cwd=str(NOTEBOOK_DIR),
    capture_output=True,
    text=True,
)
print(up.stdout.strip() or up.stderr.strip())

print("Waiting for Fuseki", end="", flush=True)
for _ in range(30):
    try:
        r = requests.get(f"{FUSEKI_BASE}/$/ping", auth=ADMIN, timeout=2)
        if r.status_code == 200:
            print(" ✓ ready!")
            break
    except Exception:
        pass
    print(".", end="", flush=True)
    time.sleep(2)
else:
    print("\nWARNING: Fuseki did not respond — verify Docker Desktop is running.")

Wrote /home/brandon/Documents/brandon/Istidama Bootcamp/Modules/Module 9A Applied Lab/docker-compose.yml
time="2026-06-07T20:36:00+03:00" level=warning msg="/home/brandon/Documents/brandon/Istidama Bootcamp/Modules/Module 9A Applied Lab/docker-compose.yml: `version` is obsolete"
 Network module9aappliedlab_default  Creating
 Network module9aappliedlab_default  Created
 Container module9aappliedlab-fuseki-1  Creating
 Container module9aappliedlab-fuseki-1  Created
 Container module9aappliedlab-fuseki-1  Starting
Error response from daemon: failed to set up container networking: driver failed programming external connectivity on endpoint module9aappliedlab-fuseki-1 (f11308222b9df74c3daa61282c0f6791f222c3a5139ada09a6e5ea8cd6fba686): Bind for 0.0.0.0:3030 failed: port is already allocated
Waiting for Fuseki ✓ ready!


## 6 · Load the Graph into Fuseki

Two steps:

1. **Create the dataset** via Fuseki's admin API (`POST /$/datasets`) — TDB2 is the
   persistent on-disk store. A 409 response means it already exists; that is fine.

2. **POST the TTL** to the Graph Store Protocol (GSP) endpoint (`POST /music/data`)
   with `Content-Type: text/turtle`.
   This is identical to what `load_dataset.py` does in the student lab.

In [8]:
# ── Create dataset ────────────────────────────────────────────────────────────
r = requests.post(
    f"{FUSEKI_BASE}/$/datasets",
    data={"dbName": DATASET, "dbType": "tdb2"},
    auth=ADMIN,
)
if r.status_code == 409:
    print(f"Dataset '{DATASET}' already exists.")
elif r.status_code == 200:
    print(f"Dataset '{DATASET}' created.")
else:
    r.raise_for_status()

# ── Upload TTL via Graph Store Protocol ───────────────────────────────────────
r = requests.post(
    f"{FUSEKI_BASE}/{DATASET}/data",
    data=MUSIC_TTL.encode("utf-8"),
    headers={"Content-Type": "text/turtle"},
    auth=ADMIN,
)
r.raise_for_status()
print(f"HTTP {r.status_code} — TTL loaded into Fuseki.")

Dataset 'music' created.
HTTP 200 — TTL loaded into Fuseki.


## 7 · Cross-Check: rdflib vs Fuseki

Run the same three queries through both engines and compare row counts.

| Engine | What it validates |
|--------|-------------------|
| **rdflib** | The Turtle is syntactically correct and the queries parse cleanly |
| **Fuseki** | Docker is up, the GSP upload succeeded, the SPARQL endpoint responds |

A count mismatch means either the upload failed silently or a query behaves
differently between the two SPARQL 1.1 engines — both are worth catching before a live demo.

In [9]:
print(f"{'Query':<30}  {'rdflib':>8}  {'Fuseki':>8}  Status")
print("-" * 60)

all_passed = True
for label, q in [("Q1 — All Albums", Q1),
                 ("Q2 — Rock (SKOS)", Q2),
                 ("Q3 — Hip-Hop ≥ 2000", Q3)]:
    local_n  = len(local_query(q))
    fuseki_n = len(fuseki_query(q))
    status   = "PASS ✓" if local_n == fuseki_n else "FAIL ✗"
    if local_n != fuseki_n:
        all_passed = False
    print(f"  {label:<28}  {local_n:>8}  {fuseki_n:>8}  {status}")

print()
print("All checks passed!" if all_passed else "One or more checks failed — see above.")
print()
print("Fuseki Q1 result preview:")
fuseki_query(Q1)

Query                             rdflib    Fuseki  Status
------------------------------------------------------------
  Q1 — All Albums                     15        15  PASS ✓
  Q2 — Rock (SKOS)                    15        15  PASS ✓
  Q3 — Hip-Hop ≥ 2000                  3         3  PASS ✓

All checks passed!

Fuseki Q1 result preview:


,title,artist,year
0,Kind of Blue,Miles Davis,1959
1,A Love Supreme,John Coltrane,1965
2,Abbey Road,The Beatles,1969
3,Led Zeppelin IV,Led Zeppelin,1971
4,The Dark Side of the Moon,Pink Floyd,1973
5,Thriller,Michael Jackson,1982
6,Purple Rain,Prince,1984
7,Nevermind,Nirvana,1991
8,The Chronic,Dr. Dre,1992
9,OK Computer,Radiohead,1997


## 8 · Shut Down Everything

Run this cell when the demo is complete.
It calls `docker compose down` which stops and removes the Fuseki container
(the named volume `fuseki_data` persists for the next run unless you add `--volumes`).

> **Tip:** If the kernel crashes before you reach this cell, open a terminal and run
> `docker compose down` from this directory manually.

In [10]:
down = subprocess.run(
    ["docker", "compose", "down"],
    cwd=str(NOTEBOOK_DIR),
    capture_output=True,
    text=True,
)
print(down.stdout.strip() or down.stderr.strip())
print("\nAll background services stopped. ✓")

time="2026-06-07T20:36:34+03:00" level=warning msg="/home/brandon/Documents/brandon/Istidama Bootcamp/Modules/Module 9A Applied Lab/docker-compose.yml: `version` is obsolete"
 Container module9aappliedlab-fuseki-1  Stopping
 Container module9aappliedlab-fuseki-1  Stopped
 Container module9aappliedlab-fuseki-1  Removing
 Container module9aappliedlab-fuseki-1  Removed
 Network module9aappliedlab_default  Removing
 Network module9aappliedlab_default  Removed

All background services stopped. ✓
